In [1]:
import pandas as pd

# Caminho do arquivo (saindo da pasta notebooks e entrando na pasta data)
caminho_csv = '../data/Basico_SP2.csv'

print("Lendo a base oficial do IBGE (isso pode levar alguns segundos, o arquivo é grande)...")
# O IBGE usa separador ';' e codificação latin1
df_sp = pd.read_csv(caminho_csv, sep=';', encoding='latin1', decimal=',')

print("Filtrando os dados apenas para Campinas...")
# O código IBGE de Campinas é 3509502
df_campinas = df_sp[df_sp['Cod_municipio'] == 3509502].copy()

# Garantindo que a coluna V005 (Rendimento nominal médio) seja lida como número
if df_campinas['V005'].dtype == 'O': # Se o pandas leu como texto
    df_campinas['V005'] = df_campinas['V005'].str.replace(',', '.').astype(float)
else:
    df_campinas['V005'] = pd.to_numeric(df_campinas['V005'], errors='coerce')

print("Agrupando a renda média por região...")
# Criando o código da região (os primeiros 10 dígitos do setor censitário)
df_campinas['regiao_ibge'] = df_campinas['Cod_setor'].astype(str).str[:10]

# Agrupando e calculando a média de renda
df_renda_regiao = df_campinas.groupby('regiao_ibge').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

print("\n✅ Tabela de Renda por Região gerada com sucesso!")
display(df_renda_regiao.sort_values(by='renda_media_bairro', ascending=False))

Lendo a base oficial do IBGE (isso pode levar alguns segundos, o arquivo é grande)...
Filtrando os dados apenas para Campinas...
Agrupando a renda média por região...

✅ Tabela de Renda por Região gerada com sucesso!


,regiao_ibge,total_setores,renda_media_bairro
4,3509502250,28,3735.332222
1,3509502100,92,3473.271848
0,3509502050,1538,2052.251930
2,3509502150,8,1980.942500
3,3509502200,57,965.423860


In [2]:
# Dicionário mapeando os códigos do IBGE para os nomes reais
dicionario_distritos = {
    '3509502050': 'Sede',
    '3509502100': 'Barão Geraldo',
    '3509502150': 'Joaquim Egídio',
    '3509502200': 'Nova Aparecida',
    '3509502250': 'Sousas'
}

# Criando a nova coluna com os nomes usando a função map()
df_renda_regiao['nome_distrito'] = df_renda_regiao['regiao_ibge'].map(dicionario_distritos)

# Reorganizando as colunas para a tabela ficar bonita
df_renda_regiao = df_renda_regiao[['regiao_ibge', 'nome_distrito', 'total_setores', 'renda_media_bairro']]

print("--- Tabela Final: Renda Média por Distrito em Campinas ---")
display(df_renda_regiao.sort_values(by='renda_media_bairro', ascending=False))

--- Tabela Final: Renda Média por Distrito em Campinas ---


,regiao_ibge,nome_distrito,total_setores,renda_media_bairro
4,3509502250,Sousas,28,3735.332222
1,3509502100,Barão Geraldo,92,3473.271848
0,3509502050,Sede,1538,2052.251930
2,3509502150,Joaquim Egídio,8,1980.942500
3,3509502200,Nova Aparecida,57,965.423860


In [4]:
# Mapeamento completo incluindo as subdivisões da Sede e os distritos
dicionario_completo = {
    '350950225000': 'Sousas',
    '350950210000': 'Barão Geraldo',
    '350950205000': 'Sede (Área Central / Maior Renda)',
    '350950215000': 'Joaquim Egídio',
    '350950205001': 'Sede (Área Periférica / Menor Renda)',
    '350950220000': 'Nova Aparecida'
}

# Criando a coluna com os nomes amigáveis
df_renda_detalhada['nome_regiao'] = df_renda_detalhada['regiao_detalhada'].map(dicionario_completo)

# Reorganizando a tabela final para apresentação
df_final_tcc = df_renda_detalhada[['regiao_detalhada', 'nome_regiao', 'total_setores', 'renda_media_bairro']]

print("--- Tabela Pronta para o TCC: Renda Média por Região em Campinas ---")
display(df_final_tcc.sort_values(by='renda_media_bairro', ascending=False))

--- Tabela Pronta para o TCC: Renda Média por Região em Campinas ---


,regiao_detalhada,nome_regiao,total_setores,renda_media_bairro
5,350950225000,Sousas,28,3735.332222
2,350950210000,Barão Geraldo,92,3473.271848
0,350950205000,Sede (Área Central / Maior Renda),996,2167.100595
3,350950215000,Joaquim Egídio,8,1980.942500
1,350950205001,Sede (Área Periférica / Menor Renda),542,1842.049207
4,350950220000,Nova Aparecida,57,965.423860


In [7]:
# Criando a tabela por cada setor censitário individual de Campinas
df_bairros = df_campinas[['Cod_setor', 'V005']].copy()
df_bairros['V005'] = pd.to_numeric(df_bairros['V005'], errors='coerce')

# Renomeando para facilitar
df_bairros.columns = ['codigo_setor', 'renda_media']

print(f"Total de setores censitários em Campinas: {len(df_bairros)}")
print("\n--- Os 10 Setores de MAIOR Renda em Campinas ---")
display(df_bairros.sort_values(by='renda_media', ascending=False).head(10))

print("\n--- Os 10 Setores de MENOR Renda em Campinas ---")
display(df_bairros.sort_values(by='renda_media', ascending=True).head(10))

Total de setores censitários em Campinas: 1723

--- Os 10 Setores de MAIOR Renda em Campinas ---


,codigo_setor,renda_media
8264,350950225000023,17417.24
6684,350950205000138,16703.64
7303,350950205000758,16656.12
8267,350950225000026,15130.43
7975,350950205001443,14044.08
7974,350950205001442,13137.85
7979,350950205001447,13034.49
7978,350950205001446,12156.74
7935,350950205001400,11746.36
8164,350950210000083,11387.49



--- Os 10 Setores de MENOR Renda em Campinas ---


,codigo_setor,renda_media
7862,350950205001323,284.84
7675,350950205001132,285.23
8080,350950205001553,300.00
7781,350950205001238,316.20
7129,350950205000584,320.83
7903,350950205001368,332.21
7857,350950205001318,362.60
7490,350950205000946,365.07
7692,350950205001149,366.23
7436,350950205000891,375.34


In [8]:
# Usando os 10 primeiros dígitos (ou o código de ponderação intermediário) para detalhar mais a cidade
df_campinas['regiao_intermediaria'] = df_campinas['Cod_setor'].astype(str).str[:9]

# Agrupando para ver todas as sub-regiões de Campinas
df_intermediario = df_campinas.groupby('regiao_intermediaria').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

print(f"Total de sub-regiões encontradas: {len(df_intermediario)}")
print("\n--- Renda Média por Sub-região de Campinas (Intermediário) ---")
display(df_intermediario.sort_values(by='renda_media_bairro', ascending=False))

Total de sub-regiões encontradas: 5

--- Renda Média por Sub-região de Campinas (Intermediário) ---


,regiao_intermediaria,total_setores,renda_media_bairro
4,350950225,28,3735.332222
1,350950210,92,3473.271848
0,350950205,1538,2052.251930
2,350950215,8,1980.942500
3,350950220,57,965.423860


In [9]:
# Usando os 11 primeiros dígitos para quebrar a Sede em várias subáreas estatísticas
df_campinas['regiao_ponderacao'] = df_campinas['Cod_setor'].astype(str).str[:11]

# Agrupando por essas subáreas
df_ponderacao = df_campinas.groupby('regiao_ponderacao').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

print(f"Total de áreas de ponderação encontradas: {len(df_ponderacao)}")
print("\n--- Renda Média por Área de Ponderação Detalhada ---")
display(df_ponderacao.sort_values(by='renda_media_bairro', ascending=False))

Total de áreas de ponderação encontradas: 5

--- Renda Média por Área de Ponderação Detalhada ---


,regiao_ponderacao,total_setores,renda_media_bairro
4,35095022500,28,3735.332222
1,35095021000,92,3473.271848
0,35095020500,1538,2052.251930
2,35095021500,8,1980.942500
3,35095022000,57,965.423860


In [10]:
# Separando a Sede e os outros distritos
df_sede = df_campinas[df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()
df_outros = df_campinas[~df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()

# Ordenando a Sede pelo código do setor para manter a proximidade geográfica
df_sede = df_sede.sort_values(by='Cod_setor').reset_index(drop=True)

# Dividindo a Sede em 10 blocos iguais (aproximadamente 150 setores cada)
df_sede['bloco_sede'] = pd.qcut(df_sede.index, q=10, labels=[f'Sede - Zona {i+1}' for i in range(10)])

# Calculando a renda média para cada uma das 10 zonas da Sede
df_blocos_sede = df_sede.groupby('bloco_sede').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index().rename(columns={'bloco_sede': 'nome_regiao'})

# Formatando os outros distritos para juntar na mesma tabela
df_outros_resumo = df_outros.groupby('Cod_setor').first().reset_index() # base para mapear
# Usando o dicionário limpo para os distritos fora da Sede
dicionario_distritos = {
    '3509502250': 'Sousas',
    '3509502100': 'Barão Geraldo',
    '3509502150': 'Joaquim Egídio',
    '3509502200': 'Nova Aparecida'
}
df_outros['nome_regiao'] = df_outros['Cod_setor'].astype(str).str[:10].map(dicionario_distritos)

df_outros_resumo = df_outros.groupby('nome_regiao').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

# Juntando a Sede fatiada com os demais distritos
df_intermediario_perfeito = pd.concat([df_blocos_sede, df_outros_resumo], ignore_index=True)

print("--- Visão Intermediária Detalhada de Campinas ---")
display(df_intermediario_perfeito.sort_values(by='renda_media_bairro', ascending=False))

--- Visão Intermediária Detalhada de Campinas ---


/tmp/ipykernel_15682/1152428650.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_blocos_sede = df_sede.groupby('bloco_sede').agg(


,nome_regiao,total_setores,renda_media_bairro
13,Sousas,28,3735.332222
10,Barão Geraldo,92,3473.271848
0,Sede - Zona 1,154,3352.616275
1,Sede - Zona 2,154,3113.355686
8,Sede - Zona 9,154,2372.357532
9,Sede - Zona 10,154,2215.657597
2,Sede - Zona 3,154,2105.750130
11,Joaquim Egídio,8,1980.942500
3,Sede - Zona 4,153,1881.841176
4,Sede - Zona 5,154,1741.741569


In [11]:
# Separando a Sede e os outros distritos
df_sede = df_campinas[df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()
df_outros = df_campinas[~df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()

# O IBGE estrutura o código do setor ordenando geograficamente os quarteirões.
# Vamos pegar um nível maior de dígitos do setor para ver as sub-regiões geográficas reais da Sede:
# Usando os primeiros 11 dígitos do setor da Sede (se houver variação) ou agrupando por faixa numérica contínua de códigos (vizinhança)
df_sede['bloco_geografico'] = df_sede['Cod_setor'].astype(str).str[:11]

# Agrupando estritamente por localização geográfica (código contínuo)
df_geo_sede = df_sede.groupby('bloco_geografico').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index().rename(columns={'bloco_geografico': 'nome_regiao'})

# Formatando os nomes das regiões geográficas da Sede com base nos blocos reais
df_geo_sede['nome_regiao'] = [f'Sede - Região Geográfica {i+1}' for i in range(len(df_geo_sede))]

# Padronizando os outros distritos
dicionario_distritos = {
    '3509502250': 'Sousas',
    '3509502100': 'Barão Geraldo',
    '3509502150': 'Joaquim Egídio',
    '3509502200': 'Nova Aparecida'
}
df_outros['nome_regiao'] = df_outros['Cod_setor'].astype(str).str[:10].map(dicionario_distritos)

df_outros_resumo = df_outros.groupby('nome_regiao').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

# Juntando tudo mantendo a coerência geográfica
df_mapa_espacial = pd.concat([df_geo_sede, df_outros_resumo], ignore_index=True)

print("--- Visão Geográfica Real por Vizinhança em Campinas ---")
display(df_mapa_espacial.sort_values(by='renda_media_bairro', ascending=False))

--- Visão Geográfica Real por Vizinhança em Campinas ---


,nome_regiao,total_setores,renda_media_bairro
4,Sousas,28,3735.332222
1,Barão Geraldo,92,3473.271848
0,Sede - Região Geográfica 1,1538,2052.251930
2,Joaquim Egídio,8,1980.942500
3,Nova Aparecida,57,965.423860


In [12]:
# Separando a Sede e os outros distritos
df_sede = df_campinas[df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()
df_outros = df_campinas[~df_campinas['Cod_setor'].astype(str).str.startswith('350950205')].copy()

# Ordenando estritamente pela numeração do setor (o que mantém a vizinhança geográfica)
df_sede = df_sede.sort_values(by='Cod_setor').reset_index(drop=True)

# Dividindo a Sede em 10 blocos geográficos sequenciais (aproximadamente 154 setores vizinhos cada)
df_sede['bloco_geo'] = pd.qcut(df_sede.index, q=10, labels=[f'Sede - Região Vizinha {i+1}' for i in range(10)])

df_geo_sede = df_sede.groupby('bloco_geo').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index().rename(columns={'bloco_geo': 'nome_regiao'})

# Padronizando os outros distritos
dicionario_distritos = {
    '3509502250': 'Sousas',
    '3509502100': 'Barão Geraldo',
    '3509502150': 'Joaquim Egídio',
    '3509502200': 'Nova Aparecida'
}
df_outros['nome_regiao'] = df_outros['Cod_setor'].astype(str).str[:10].map(dicionario_distritos)

df_outros_resumo = df_outros.groupby('nome_regiao').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

# Juntando a Sede fatiada por vizinhança com os demais distritos
df_espacial_real = pd.concat([df_geo_sede, df_outros_resumo], ignore_index=True)

print("--- Visão Espacial Coesa por Vizinhança em Campinas ---")
display(df_espacial_real.sort_values(by='renda_media_bairro', ascending=False))

--- Visão Espacial Coesa por Vizinhança em Campinas ---


/tmp/ipykernel_15682/1118671743.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_geo_sede = df_sede.groupby('bloco_geo').agg(


,nome_regiao,total_setores,renda_media_bairro
13,Sousas,28,3735.332222
10,Barão Geraldo,92,3473.271848
0,Sede - Região Vizinha 1,154,3352.616275
1,Sede - Região Vizinha 2,154,3113.355686
8,Sede - Região Vizinha 9,154,2372.357532
9,Sede - Região Vizinha 10,154,2215.657597
2,Sede - Região Vizinha 3,154,2105.750130
11,Joaquim Egídio,8,1980.942500
3,Sede - Região Vizinha 4,153,1881.841176
4,Sede - Região Vizinha 5,154,1741.741569


In [15]:
# Quebrando a Sede e os distritos pelas Áreas de Ponderação oficiais (11 dígitos do Censo)
df_campinas['area_ponderacao'] = df_campinas['Cod_setor'].astype(str).str[:11]

# Agrupando por área de ponderação para detalhar a Sede e os demais distritos
df_detalhado_oficial = df_campinas.groupby('area_ponderacao').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

# Dicionário de nomeação para as macro-regiões e áreas da Sede e distritos
# Os códigos com final 050 derivam da Sede, os demais dos distritos periféricos
def rotular_regiao(codigo):
    if codigo.startswith('350950225'):
        return 'Sousas'
    elif codigo.startswith('350950210'):
        return 'Barão Geraldo'
    elif codigo.startswith('350950215'):
        return 'Joaquim Egídio'
    elif codigo.startswith('350950220'):
        return 'Nova Aparecida'
    elif codigo.startswith('350950205'):
        # Identificando as fatias oficiais da Sede
        return f'Sede - Área de Ponderação {codigo[-2:]}'
    else:
        return 'Outras Áreas'

df_detalhado_oficial['nome_regiao'] = df_detalhado_oficial['area_ponderacao'].apply(rotular_regiao)

print("--- Tabela Detalhada por Áreas de Ponderação Oficiais do IBGE ---")
display(df_detalhado_oficial[['area_ponderacao', 'nome_regiao', 'total_setores', 'renda_media_bairro']].sort_values(by='renda_media_bairro', ascending=False))

--- Tabela Detalhada por Áreas de Ponderação Oficiais do IBGE ---


,area_ponderacao,nome_regiao,total_setores,renda_media_bairro
4,35095022500,Sousas,28,3735.332222
1,35095021000,Barão Geraldo,92,3473.271848
0,35095020500,Sede - Área de Ponderação 00,1538,2052.251930
2,35095021500,Joaquim Egídio,8,1980.942500
3,35095022000,Nova Aparecida,57,965.423860
